# SE446 Week 10A — Kafka Connection Test

**Course:** SE446 Big Data Engineering — Alfaisal University
**Topic:** Apache Kafka — Verify Your Setup

This notebook tests your connection to the Kafka broker on the cluster.
It creates a **secure SSH tunnel** using your cluster credentials.

Works on **Google Colab**, **local Jupyter**, and **VS Code**.

---

## 1. Install the Kafka Python Client

In [ ]:
!pip install confluent-kafka "sshtunnel>=0.4.0" "paramiko>=3.5,<4" -q

## 2. Connect to the Cluster

Enter your **cluster SSH credentials** below. The notebook creates a secure tunnel — all Kafka traffic is encrypted.

Set your `STUDENT_ID` to create a personal topic.

In [ ]:
STUDENT_ID = "YOUR_NAME"     # <-- CHANGE THIS

import os, json, uuid, getpass
from sshtunnel import SSHTunnelForwarder

MASTER_IP = "134.209.172.50"

if 'tunnel' in dir() and tunnel.is_active:
    print("SSH tunnel already active — reusing.")
else:
    print(f"=== SSH Tunnel to {MASTER_IP} ===")
    ssh_user = input("SSH Username: ")
    ssh_pass = getpass.getpass("SSH Password: ")

    tunnel = SSHTunnelForwarder(
        (MASTER_IP, 22),
        ssh_username=ssh_user,
        ssh_password=ssh_pass,
        remote_bind_address=("localhost", 9092),
        local_bind_address=("localhost", 9092),
    )
    tunnel.start()
    print(f"Tunnel open: localhost:{tunnel.local_bind_port} → {MASTER_IP}:9092")

BROKER = f"localhost:{tunnel.local_bind_port}"
TOPIC = f"test-{STUDENT_ID}"

print(f"\nBroker:  {BROKER}")
print(f"Topic:   {TOPIC}")

## 3. Test 1 — Connection Check

Try to connect to the broker and list existing topics. If this cell fails, your tunnel or network is not working.

In [ ]:
from confluent_kafka.admin import AdminClient

admin = AdminClient({"bootstrap.servers": BROKER})

try:
    metadata = admin.list_topics(timeout=10)
    print(f"Connected to Kafka broker at {BROKER}")
    print(f"Cluster ID: {metadata.cluster_id}")
    print(f"Brokers:    {list(metadata.brokers.values())}")
    print(f"\nExisting topics ({len(metadata.topics)}):")
    for name in sorted(metadata.topics):
        if not name.startswith("__"):  # skip internal topics
            t = metadata.topics[name]
            print(f"  {name:30s} ({len(t.partitions)} partitions)")
    print("\n--- CONNECTION TEST: PASSED ---")
except Exception as e:
    print(f"--- CONNECTION TEST: FAILED ---")
    print(f"Error: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Is your SSH tunnel running?  ssh -L 9092:localhost:9092 user@MASTER_IP")
    print(f"  2. Is Kafka running on the server?  systemctl status kafka")
    print(f"  3. Is the broker address correct?  Currently: {BROKER}")

## 4. Test 2 — Create a Topic

Create your personal topic with 3 partitions. This is safe to re-run — it will just report the topic already exists.

In [ ]:
from confluent_kafka.admin import NewTopic

new_topic = NewTopic(TOPIC, num_partitions=3, replication_factor=1)
futures = admin.create_topics([new_topic])

for topic_name, future in futures.items():
    try:
        future.result()  # blocks until topic is created
        print(f"Topic '{topic_name}' created with 3 partitions")
    except Exception as e:
        if "already exists" in str(e):
            print(f"Topic '{topic_name}' already exists (OK)")
        else:
            print(f"Failed to create topic '{topic_name}': {e}")

# Verify
meta = admin.list_topics(timeout=10)
if TOPIC in meta.topics:
    t = meta.topics[TOPIC]
    print(f"\nVerified: '{TOPIC}' has {len(t.partitions)} partitions")
    print("--- TOPIC CREATION TEST: PASSED ---")

## 5. Test 3 — Produce Messages

Send 5 crime events to your topic. Each message has a **key** (district) and a **value** (JSON payload).

Watch which partition each message lands on — same key always goes to the same partition.

In [ ]:
from confluent_kafka import Producer
import json

producer = Producer({"bootstrap.servers": BROKER})

# Sample crime events
crimes = [
    {"id": 1, "type": "THEFT",       "district": 8,  "arrest": False, "hour": 14},
    {"id": 2, "type": "BATTERY",     "district": 11, "arrest": True,  "hour": 22},
    {"id": 3, "type": "ROBBERY",     "district": 3,  "arrest": False, "hour": 2},
    {"id": 4, "type": "ASSAULT",     "district": 8,  "arrest": True,  "hour": 19},
    {"id": 5, "type": "BURGLARY",    "district": 3,  "arrest": False, "hour": 4},
]

# Delivery callback — called once per message when broker acknowledges
def on_delivery(err, msg):
    if err:
        print(f"  FAILED: {err}")
    else:
        print(f"  Delivered to partition {msg.partition()} | offset {msg.offset()} | key={msg.key().decode()}")

print(f"Producing {len(crimes)} messages to '{TOPIC}'...\n")
for crime in crimes:
    producer.produce(
        topic=TOPIC,
        key=str(crime["district"]),                # key = district (determines partition)
        value=json.dumps(crime).encode("utf-8"),   # value = JSON bytes
        callback=on_delivery
    )

# flush() blocks until all messages are delivered (or fail)
remaining = producer.flush(timeout=10)
print(f"\nMessages remaining in queue: {remaining}")
print("--- PRODUCER TEST: PASSED ---" if remaining == 0 else "--- PRODUCER TEST: SOME MESSAGES FAILED ---")

### Observe the Partition Assignment

Notice that:
- Messages with **key="8"** (district 8) always go to the **same partition**
- Messages with **key="3"** (district 3) always go to the **same partition**
- Messages with **key="11"** go to a **different partition**

This is **hash partitioning**: `hash(key) % 3` determines the partition.

## 6. Test 4 — Consume Messages

Read back all messages from the beginning. The consumer uses:
- `group.id` — identifies this consumer group
- `auto.offset.reset = earliest` — start from offset 0 (replay all)

In [ ]:
from confluent_kafka import Consumer
import json

consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"{STUDENT_ID}-test-group",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,          # manual control for clarity
})

consumer.subscribe([TOPIC])

print(f"Consuming from '{TOPIC}' (waiting up to 10 seconds)...\n")
print(f"{'Partition':>10} {'Offset':>8} {'Key':>6}   {'Value'}")
print("-" * 70)

messages = []
empty_polls = 0

while empty_polls < 10:
    msg = consumer.poll(timeout=1.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        print(f"Error: {msg.error()}")
        continue
    
    empty_polls = 0  # reset on successful read
    key = msg.key().decode() if msg.key() else "None"
    value = msg.value().decode()
    crime = json.loads(value)
    messages.append(crime)
    
    print(f"{msg.partition():>10} {msg.offset():>8} {key:>6}   {crime['type']:12s} district={crime['district']}")

consumer.close()

print(f"\n Total messages consumed: {len(messages)}")
print("--- CONSUMER TEST: PASSED ---" if len(messages) >= 5 else "--- CONSUMER TEST: CHECK OUTPUT ---")

## 7. Test 5 — Replay (Re-read the Same Messages)

One of Kafka's key features: messages are **retained, not deleted** after reading.
Let's prove it by reading them again with a **new consumer group**.

In [ ]:
import uuid

# Use a random group ID so Kafka treats this as a brand-new consumer
replay_consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"replay-{uuid.uuid4().hex[:8]}",   # unique group = fresh start
    "auto.offset.reset": "earliest",
})

replay_consumer.subscribe([TOPIC])

print("Replaying all messages with a new consumer group...\n")

replay_count = 0
empty = 0
while empty < 10:
    msg = replay_consumer.poll(timeout=1.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    replay_count += 1

replay_consumer.close()

print(f"Replayed {replay_count} messages (same data, new consumer group)")
print()
if replay_count >= 5:
    print("This proves Kafka RETAINS messages after reading.")
    print("In RabbitMQ, the messages would be gone after the first consumer read them.")
    print("\n--- REPLAY TEST: PASSED ---")
else:
    print("--- REPLAY TEST: CHECK OUTPUT ---")

## 8. Test 6 — Topic Info and Partition Details

Inspect your topic: how many partitions, what offsets exist, and the broker leader for each partition.

In [ ]:
from confluent_kafka import TopicPartition

meta = admin.list_topics(timeout=10)
topic_meta = meta.topics[TOPIC]

print(f"Topic: {TOPIC}")
print(f"Partitions: {len(topic_meta.partitions)}")
print()

# Get the latest offset (high watermark) for each partition
tmp_consumer = Consumer({
    "bootstrap.servers": BROKER,
    "group.id": f"info-{uuid.uuid4().hex[:8]}",
})

print(f"{'Partition':>10} {'Leader':>8} {'Messages':>10}")
print("-" * 35)

for pid in sorted(topic_meta.partitions):
    p = topic_meta.partitions[pid]
    # Get watermark offsets (low, high)
    lo, hi = tmp_consumer.get_watermark_offsets(TopicPartition(TOPIC, pid), timeout=5)
    print(f"{pid:>10} {p.leader:>8} {hi - lo:>10}")

tmp_consumer.close()
print("\n--- TOPIC INFO TEST: PASSED ---")

## 9. Cleanup (Optional)

Delete your test topic when you're done. This is optional — the topic will be auto-deleted after 7 days (retention policy).

In [ ]:
# Uncomment the lines below to delete your test topic
# futures = admin.delete_topics([TOPIC])
# for topic_name, future in futures.items():
#     try:
#         future.result()
#         print(f"Topic '{topic_name}' deleted")
#     except Exception as e:
#         print(f"Failed: {e}")

---

## Summary of Tests

| # | Test | What It Proves |
|---|------|---------------|
| 1 | Connection check | Broker is reachable, cluster is alive |
| 2 | Create topic | You can create topics with partitions |
| 3 | Produce messages | Producer sends keyed messages, hash partitioning works |
| 4 | Consume messages | Consumer reads from all partitions, offsets track correctly |
| 5 | Replay | Messages are retained (not deleted) — Kafka is not a queue |
| 6 | Topic info | You can inspect partition layout and message counts |

If all tests pass, your Kafka setup is working correctly and you are ready for the lab.

**Next:** Session 10B — Connect this to Spark Structured Streaming with `spark.readStream.format("kafka")`